# Qwen LoRA fine-tuning

LRZ: https://login.ai.lrz.de (GPU session required)

1. Run the dependency cell, then restart the kernel if packages changed
2. Paste `HF_TOKEN` in the model cell (optional for public models)
3. In the config cell: set `TRAIN_DATA_SPLIT`, `DATA_SPLIT`, `DATA_DIR`, and sample limits if needed
4. Run the LoRA training cell to save the adapter, then run the evaluation cell

Inputs: training data from `Finetuning experiment/data/training/`; evaluation defaults to `Finetuning experiment/data/test/`.

Outputs: LoRA adapter in `qwen_lora_adapter/`; predictions and metrics in `qwen_lora_translation/` under the evaluation data directory.

In [14]:
# Pin transformers: LRZ PyTorch (nv24.08) lacks torch.float8_e8m0fnu required by transformers>=4.51.
%pip install -q "transformers>=4.46,<4.51" accelerate peft sacrebleu tqdm huggingface_hub ipywidgets
print("Restart kernel, then run from the config cell.")

Note: you may need to restart the kernel to use updated packages.
Restart kernel, then run from the config cell.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\vnpnk\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import os

WORK_DIR = Path.cwd()

# --- user settings ---

# Evaluation split: "training", "validation", or "test"
DATA_SPLIT = "test"

# LoRA training split. Keep this as training to avoid fitting on validation/test.
TRAIN_DATA_SPLIT = "training"

# None = auto-detect selected evaluation split; set explicitly if needed
DATA_DIR = None

# None = auto-detect selected training split; set explicitly if needed
TRAIN_DATA_DIR = None

# Evaluation samples. None = all; e.g. 10 for a quick smoke test
MAX_SAMPLES = 50

# Training samples. None = all training examples.
MAX_TRAIN_SAMPLES = None

# --- LoRA training settings ---
LORA_OUTPUT_DIR = "qwen_lora_adapter"
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_LEARNING_RATE = 2e-4
LORA_NUM_TRAIN_EPOCHS = 1
LORA_PER_DEVICE_TRAIN_BATCH_SIZE = 1
LORA_GRADIENT_ACCUMULATION_STEPS = 8
LORA_MAX_SEQ_LENGTH = 1024

LANG_GROUPS = ["ende", "enru", "enes"]
MODES = ["proper_term"]

LANG_CONFIG = {
    "ende": {"ref_field": "de", "target_lang": "German", "output_tag": "de"},
    "enru": {"ref_field": "ru", "target_lang": "Russian", "output_tag": "ru"},
    "enes": {"ref_field": "es", "target_lang": "Spanish", "output_tag": "es"},
}
# Few-shot examples from *_dev_v2.jsonl.
SAMPLE_SENTENCES: dict[str, list[dict[str, object]]] = {
    "ende": [
        {"en": "The status of each individual space can be seen from the color code on the upper left corner.", "de": "Der Farbcode oben links gibt den Status des betreffenden Space an.", "proper_terms": {"space": "Space"}, "random_terms": {"status": "Status"}},
        {"en": "This service describes the deployed (run-time) state of SAP HANA database artifacts, for example: tables, views, or procedures, which have been created or adjusted by the SAP Integrated Development Environment (WebIDE) editors as a family of consistent design-time artifacts for all key SAP HANA platform database features.", "de": "Dieser Service beschreibt den implementierten Zustand (Laufzeitzustand) von SAP-HANA-Datenbankartefakten, z.Â B. Tabellen, Views oder Prozeduren, die von den SAP-Integrated-Development-Environment-Editoren (WebIDE-Editoren) als eine Familie konsistenter Entwurfszeit-Artefakte fÃ¼r alle wichtigen SAP-HANA-Plattform-Datenbankfunktionen erstellt oder angepasst wurden.", "proper_terms": {"design": "Entwurf", "state": "Zustand"}, "random_terms": {"service": "Service", "features": "funktionen"}},
        {"en": "Your contract includes a total number of available capacity units per month which you can allocate as you wish to the compute and storage resources.", "de": "Ihr Vertrag enthÃ¤lt eine Gesamtzahl verfÃ¼gbarer KapazitÃ¤tseinheiten pro Monat, die Sie den Rechen- und Speicherressourcen nach Belieben zuweisen kÃ¶nnen.", "proper_terms": {"storage": "Speicher"}, "random_terms": {"total": "Gesamtzahl", "available": "verfÃ¼gbarer"}}
    ],
    "enru": [
        {"en": "To add notes, choose a template and open the Notes pane on the left side of the screen.", "ru": "Ð§Ñ‚Ð¾Ð±Ñ‹ Ð´Ð¾Ð±Ð°Ð²Ð¸Ñ‚ÑŒ Ð¿Ñ€Ð¸Ð¼ÐµÑ‡Ð°Ð½Ð¸Ñ, Ð²Ñ‹Ð±ÐµÑ€Ð¸Ñ‚Ðµ ÑˆÐ°Ð±Ð»Ð¾Ð½ Ð¸ Ð¾Ñ‚ÐºÑ€Ð¾Ð¹Ñ‚Ðµ Ð¾Ð±Ð»Ð°ÑÑ‚ÑŒ ÐŸÑ€Ð¸Ð¼ÐµÑ‡Ð°Ð½Ð¸Ñ Ð½Ð° Ð»ÐµÐ²Ð¾Ð¹ ÑÑ‚Ð¾Ñ€Ð¾Ð½Ðµ ÑÐºÑ€Ð°Ð½Ð°.", "proper_terms": {"pane": "Ð¾Ð±Ð»Ð°ÑÑ‚ÑŒ"}, "random_terms": {"add": "Ð´Ð¾Ð±Ð°Ð²Ð¸Ñ‚ÑŒ", "choose": "Ð²Ñ‹Ð±ÐµÑ€Ð¸Ñ‚Ðµ"}},
        {"en": "Indicates if a configuration item or configuration step is specific to a localized solution version.", "ru": "Ð£ÐºÐ°Ð·Ñ‹Ð²Ð°ÐµÑ‚, ÑÐ²Ð»ÑÑŽÑ‚ÑÑ Ð»Ð¸ Ð¿Ð¾Ð·Ð¸Ñ†Ð¸Ñ Ð¸Ð»Ð¸ ÑˆÐ°Ð³ ÐºÐ¾Ð½Ñ„Ð¸Ð³ÑƒÑ€Ð°Ñ†Ð¸Ð¸ ÑÐ¿ÐµÑ†Ð¸Ñ„Ð¸Ñ‡Ð½Ñ‹Ð¼Ð¸ Ð´Ð»Ñ Ð»Ð¾ÐºÐ°Ð»Ð¸Ð·Ð¾Ð²Ð°Ð½Ð½Ð¾Ð¹ Ð²ÐµÑ€ÑÐ¸Ð¸ Ñ€ÐµÑˆÐµÐ½Ð¸Ñ.", "proper_terms": {"item": "Ð¿Ð¾Ð·Ð¸Ñ†Ð¸Ñ"}, "random_terms": {"version": "Ð²ÐµÑ€ÑÐ¸Ð¸", "solution": "Ñ€ÐµÑˆÐµÐ½Ð¸Ñ"}},
        {"en": "Specifies the number of the contract from which you can select service items.", "ru": "Ð£ÐºÐ°Ð·Ñ‹Ð²Ð°ÐµÑ‚ Ð½Ð¾Ð¼ÐµÑ€ ÐºÐ¾Ð½Ñ‚Ñ€Ð°ÐºÑ‚Ð°, Ð¸Ð· ÐºÐ¾Ñ‚Ð¾Ñ€Ð¾Ð³Ð¾ Ð¼Ð¾Ð¶Ð½Ð¾ Ð²Ñ‹Ð±Ñ€Ð°Ñ‚ÑŒ Ð¿Ð¾Ð·Ð¸Ñ†Ð¸Ð¸ ÑƒÑÐ»ÑƒÐ³.", "proper_terms": {"contract": "ÐºÐ¾Ð½Ñ‚Ñ€Ð°ÐºÑ‚"}, "random_terms": {"number": "Ð½Ð¾Ð¼ÐµÑ€", "select": "Ð²Ñ‹Ð±Ñ€Ð°Ñ‚ÑŒ"}}
    ],
    "enes": [
        {"en": "Why would you need to access HDI containers?", "es": "Â¿Por quÃ© tendrÃ­a que acceder a los containers HDI?", "proper_terms": {"container": "container"}, "random_terms": {"access": "acceder"}},
        {"en": "In such cases you may use the Move Items or Merge feature.", "es": "En estos casos, puede utilizar la funciÃ³n Mover elementos o Fusionar .", "proper_terms": {"item": "elemento"}, "random_terms": {"may": "puede"}},
        {"en": "Decide if you want to use parallel processing for this job:", "es": "Decida si desea utilizar el procesamiento paralelo para este job:", "proper_terms": {"processing": "procesamiento", "job": "job", "parallel processing": "procesamiento paralelo"}, "random_terms": {"Decide": "Decida", "want": "desea"}}
    ],
}


def split_suffix(split: str | None = None) -> str:
    normalized = (split or DATA_SPLIT).strip().lower()
    if normalized not in {"training", "validation", "test"}:
        raise ValueError("split must be one of: training, validation, test")
    return normalized


def data_stem(lang: str, split: str | None = None) -> str:
    return f"{lang}_dev_v1_{split_suffix(split)}"


def _data_dir_candidates(split: str | None = None, data_dir: str | Path | None = None) -> list[Path]:
    selected_split = split_suffix(split)
    home = Path.home()
    env_home = Path(os.environ["HOME"]).expanduser() if os.environ.get("HOME") else None
    repo_roots = [WORK_DIR, home, WORK_DIR.parent]
    if env_home is not None:
        repo_roots.append(env_home)
    if WORK_DIR.is_dir():
        repo_roots.extend(p for p in WORK_DIR.iterdir() if p.is_dir())
    candidates: list[Path] = []
    if data_dir is not None:
        candidates.append(Path(data_dir).expanduser())
    for base in repo_roots:
        candidates.extend(
            [
                base / "Finetuning experiment" / "data" / selected_split,
                base / "data" / selected_split,
            ]
        )
    seen: set[Path] = set()
    unique: list[Path] = []
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def resolve_data_dir(split: str | None = None, data_dir: str | Path | None = None) -> Path:
    selected_split = split_suffix(split)
    probe = f"{data_stem('ende', selected_split)}.jsonl"
    for candidate in _data_dir_candidates(selected_split, data_dir):
        if (candidate / probe).exists():
            return candidate
    fallback = Path(data_dir).expanduser() if data_dir is not None else WORK_DIR / "data" / selected_split
    return fallback.resolve()


DATA_ROOT = resolve_data_dir(DATA_SPLIT, DATA_DIR)
TRAIN_DATA_ROOT = resolve_data_dir(TRAIN_DATA_SPLIT, TRAIN_DATA_DIR)
OUTPUT_BASE = (DATA_ROOT / "qwen_lora_translation").resolve()
ADAPTER_DIR = (WORK_DIR / LORA_OUTPUT_DIR).resolve()


def data_path(lang: str, split: str | None = None, data_root: Path | None = None) -> Path:
    root = data_root or DATA_ROOT
    return root / f"{data_stem(lang, split)}.jsonl"


def prediction_stem(lang: str) -> str:
    return data_stem(lang, DATA_SPLIT)


print("TRAIN_DATA_SPLIT:", split_suffix(TRAIN_DATA_SPLIT))
print("DATA_SPLIT:", split_suffix(DATA_SPLIT))
print("KERNEL_CWD:", WORK_DIR)
print("TRAIN_DATA_DIR:", TRAIN_DATA_ROOT)
print("DATA_DIR:", DATA_ROOT)
if DATA_ROOT != WORK_DIR.resolve() or TRAIN_DATA_ROOT != WORK_DIR.resolve():
    print("Note: data found outside kernel cwd (common on LRZ). Outputs go next to evaluation data files.")
print("MAX_TRAIN_SAMPLES:", "all" if MAX_TRAIN_SAMPLES is None else MAX_TRAIN_SAMPLES)
print("MAX_SAMPLES:", "all" if MAX_SAMPLES is None else MAX_SAMPLES)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("OUTPUT_BASE:", OUTPUT_BASE)
for split_name, root in [("train", TRAIN_DATA_ROOT), ("eval", DATA_ROOT)]:
    selected_split = TRAIN_DATA_SPLIT if split_name == "train" else DATA_SPLIT
    for lang in LANG_GROUPS:
        path = data_path(lang, selected_split, root)
        status = "ok" if path.exists() else "MISSING"
        print(f"  {split_name}/{lang}: {path.name} [{status}] ({len(SAMPLE_SENTENCES[lang])} prompt examples)")

In [ ]:
import os
from pathlib import Path

import torch

# LRZ containers ship an older PyTorch build; recent transformers import FP8 dtypes at load time.
for _fp8_name in ("float8_e8m0fnu", "float8_e4m3fn", "float8_e5m2"):
    if not hasattr(torch, _fp8_name):
        setattr(torch, _fp8_name, torch.float32)

from transformers import AutoModelForCausalLM, AutoTokenizer

HF_TOKEN = "***HF_TOKEN_REDACTED***"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()


def setup_hf_cache(root: Path) -> Path:
    if os.environ.get("HF_HOME"):
        cache_root = Path(os.environ["HF_HOME"]).expanduser()
    elif os.environ.get("SCRATCH"):
        cache_root = Path(os.environ["SCRATCH"]) / "huggingface_cache"
    else:
        cache_root = root / ".cache" / "huggingface"
    cache_root.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache_root)
    os.environ["HUGGINGFACE_HUB_CACHE"] = str(cache_root / "hub")
    return cache_root


HF_CACHE = setup_hf_cache(Path.cwd())
MODEL_NAME = os.environ.get("QWEN_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
LOCAL_MODEL_DIR = os.environ.get("QWEN_MODEL_DIR", "").strip()
model_source = LOCAL_MODEL_DIR or MODEL_NAME

print("HF cache:", HF_CACHE)
print("torch:", torch.__version__)
if torch.cuda.is_available():
    print("CUDA:", torch.cuda.get_device_name(0))
else:
    print("Warning: no GPU â€” CPU only (very slow).")

load_kwargs = {
    "torch_dtype": "auto",
    "cache_dir": str(HF_CACHE),
    "device_map": "auto" if torch.cuda.is_available() else "cpu",
}

print(f"Loading model: {model_source}")
model = AutoModelForCausalLM.from_pretrained(model_source, **load_kwargs)
tokenizer = AutoTokenizer.from_pretrained(model_source, cache_dir=str(HF_CACHE))
print("Model ready.")

HF cache: c:\Users\vnpnk\Documents\TUM\studies\semester 2\Terminology translation\terminology-translation\Baseline\.cache\huggingface
torch: 2.12.0+cpu
Loading model: Qwen/Qwen2.5-3B-Instruct


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model ready.


In [3]:
import json
import re
from collections import Counter, defaultdict
from typing import Any

import sacrebleu
from tqdm import tqdm


# --- I/O ---

def load_jsonl(path: Path, max_samples: int | None = None) -> list[dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
            if max_samples is not None and len(records) >= max_samples:
                break
    return records


def save_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_datasets(
    split: str | None = None,
    data_root: Path | None = None,
    max_samples: int | None = MAX_SAMPLES,
) -> dict[str, list[dict[str, Any]]]:
    selected_split = split or DATA_SPLIT
    root = data_root or DATA_ROOT
    datasets = {}
    for lang in LANG_GROUPS:
        path = data_path(lang, selected_split, root)
        if not path.exists():
            raise FileNotFoundError(f"Missing dataset for {lang}: {path}")
        datasets[lang] = load_jsonl(path, max_samples)
    return datasets


# --- terminology helpers ---

def terms_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str]:
    if mode == "proper_term":
        return (sample.get("proper_terms") or {}).copy()
    raise ValueError(f"Unsupported mode: {mode}. This notebook runs only proper_term mode.")


def terminology_for_mode(sample: dict[str, Any], mode: str) -> dict[str, str] | None:
    terms = terms_for_mode(sample, mode)
    return terms or None


def strip_output_tags(text: str, output_tag: str) -> str:
    if not isinstance(text, str):
        return text
    return re.sub(rf"</?{re.escape(output_tag)}>", "", text, flags=re.IGNORECASE).strip()


# --- metrics ---

def compute_bleu_chrf(hyps: list[str], refs: list[str]) -> dict[str, float]:
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    return {"bleu": bleu.score, "chrf": chrf.score}


def _normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())


def _count_term_occurrences(text: str, term: str) -> int:
    text_norm = _normalize_text(text)
    term_norm = _normalize_text(term)
    return len(re.findall(r"\b" + re.escape(term_norm) + r"\b", text_norm))


def terminology_accuracy(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_ratios: dict[str, float] = {}
    total_terms = 0

    for pred, sample in zip(preds, samples):
        source_text = sample.get("en", "")
        for src, tgt in terms_for_mode(sample, mode).items():
            total_terms += 1
            src_count = max(_count_term_occurrences(source_text, src), 1)
            tgt_count = _count_term_occurrences(pred, tgt)
            term_ratios[src] = min(tgt_count / src_count, 1.0)

    avg_ratio = sum(term_ratios.values()) / len(term_ratios) * 100 if term_ratios else None
    return {"total_terms": total_terms, "avg_ratio_pct": avg_ratio, "per_term_ratios": term_ratios}


def terminology_consistency(preds: list[str], samples: list[dict[str, Any]], mode: str) -> dict[str, Any]:
    term_to_candidates: dict[str, list[str]] = defaultdict(list)

    for pred, sample in zip(preds, samples):
        for src, tgt in terms_for_mode(sample, mode).items():
            candidate = tgt if str(tgt).lower() in str(pred).lower() else "<MISSING>"
            term_to_candidates[src].append(candidate)

    per_term = {}
    macro_scores = []
    weighted_scores = []

    for src, candidates in term_to_candidates.items():
        pseudo_ref = Counter(candidates).most_common(1)[0][0]
        matches = sum(1 for c in candidates if c == pseudo_ref)
        consistency = matches / len(candidates)
        per_term[src] = {
            "occ": len(candidates),
            "pseudo_ref": pseudo_ref,
            "matches": matches,
            "consistency": consistency,
        }
        macro_scores.append(consistency)
        weighted_scores.extend([consistency] * len(candidates))

    return {
        "per_term": per_term,
        "macro_avg_consistency": sum(macro_scores) / len(macro_scores) if macro_scores else None,
        "weighted_avg_consistency": sum(weighted_scores) / len(weighted_scores) if weighted_scores else None,
    }


def fmt_metric(value: float | None, digits: int = 2) -> str:
    return "N/A" if value is None else f"{value:.{digits}f}"


# --- translation ---

def format_terminology_block(terms: dict[str, str]) -> str:
    if not terms:
        return ""
    return "Terminology:\n" + "\n".join(f"{s} -> {t}" for s, t in terms.items()) + "\n"


def format_sample_examples(lang: str, mode: str) -> str:
    config = LANG_CONFIG[lang]
    ref_field = config["ref_field"]
    output_tag = config["output_tag"]
    blocks = []
    for i, example in enumerate(SAMPLE_SENTENCES[lang], 1):
        term_block = format_terminology_block(terms_for_mode(example, mode))
        ref = example.get(ref_field, "")
        blocks.append(
            f"Example {i}:\n"
            f"{term_block}"
            f"Input:\n<en> {example['en']} </en>\n"
            f"Output:\n<{output_tag}> {ref} </{output_tag}>"
        )
    return "Examples:\n\n" + "\n\n".join(blocks) + "\n\n"


def build_translation_prompt(
    sample_en: str,
    terminology: dict[str, str] | None,
    target_lang: str,
    output_tag: str,
    lang: str,
    mode: str,
) -> str:
    examples_block = format_sample_examples(lang, mode)
    term_block = format_terminology_block(terminology or {})
    if term_block:
        term_block += "\n"

    return f"""You are a translation assistant.

Translate the English text to {target_lang}.

Rules:
1. Output only in this format: <{output_tag}> ... </{output_tag}>
2. Use the terminology mappings exactly as provided.
3. Do not explain anything.
4. Translate only from English to {target_lang}.

{examples_block}{term_block}Input:
<en> {sample_en} </en>
"""


def translate_sample(
    sample_en: str,
    terminology: dict[str, str] | None,
    target_lang: str,
    output_tag: str,
    lang: str,
    mode: str,
    max_new_tokens: int = 256,
) -> str:
    prompt = build_translation_prompt(sample_en, terminology, target_lang, output_tag, lang, mode)
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    generated_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model


class TranslationPromptDataset(Dataset):
    def __init__(self, datasets_by_lang: dict[str, list[dict[str, Any]]], mode: str = "proper_term"):
        self.items: list[tuple[str, dict[str, Any]]] = []
        for lang, samples in datasets_by_lang.items():
            self.items.extend((lang, sample) for sample in samples)
        self.mode = mode

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> dict[str, list[int]]:
        lang, sample = self.items[idx]
        config = LANG_CONFIG[lang]
        ref_field = config["ref_field"]
        output_tag = config["output_tag"]
        prompt = build_translation_prompt(
            sample.get("en", ""),
            terminology_for_mode(sample, self.mode),
            config["target_lang"],
            output_tag,
            lang,
            self.mode,
        )
        target = f"<{output_tag}> {sample.get(ref_field, '')} </{output_tag}>"
        prompt_messages = [
            {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ]
        full_messages = prompt_messages + [{"role": "assistant", "content": target}]
        prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
        encoded = tokenizer(full_text, truncation=True, max_length=LORA_MAX_SEQ_LENGTH)
        prompt_ids = tokenizer(prompt_text, truncation=True, max_length=LORA_MAX_SEQ_LENGTH)["input_ids"]
        labels = encoded["input_ids"].copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len
        encoded["labels"] = labels
        return encoded


def causal_lm_collator(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    max_len = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad_len = max_len - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad_len)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad_len)
        batch["labels"].append(feature["labels"] + [-100] * pad_len)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_datasets = load_datasets(TRAIN_DATA_SPLIT, TRAIN_DATA_ROOT, MAX_TRAIN_SAMPLES)
validation_datasets = load_datasets("validation", resolve_data_dir("validation"), MAX_SAMPLES)
train_dataset = TranslationPromptDataset(train_datasets)
validation_dataset = TranslationPromptDataset(validation_datasets)

bf16_enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_enabled = torch.cuda.is_available() and not bf16_enabled
training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR / "trainer_checkpoints"),
    num_train_epochs=LORA_NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=LORA_PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=LORA_GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LORA_LEARNING_RATE,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=bf16_enabled,
    fp16=fp16_enabled,
    gradient_checkpointing=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=causal_lm_collator,
)
trainer.train()
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
model.config.use_cache = True
print("LoRA adapter saved to:", ADAPTER_DIR)

In [ ]:
def prediction_filename(lang: str, mode: str) -> str:
    return f"{prediction_stem(lang)}_{mode}_predictions.jsonl"


def run_mode(
    lang: str,
    mode: str,
    samples: list[dict[str, Any]],
    output_dir: Path,
    config: dict[str, str],
) -> dict[str, Any]:
    ref_field = config["ref_field"]
    preds = []
    records = []

    for sample in tqdm(samples, desc=f"{lang}/{mode}"):
        pred = translate_sample(
            sample.get("en", ""),
            terminology_for_mode(sample, mode),
            config["target_lang"],
            config["output_tag"],
            lang,
            mode,
        )
        preds.append(pred)
        record = sample.copy()
        record[f"prediction_{mode}"] = pred
        record[f"prediction_{mode}_clean"] = strip_output_tags(pred, config["output_tag"])
        records.append(record)

    clean_preds = [strip_output_tags(p, config["output_tag"]) for p in preds]
    pred_path = output_dir / prediction_filename(lang, mode)
    save_jsonl(pred_path, records)

    metrics: dict[str, Any] = {}
    if samples and ref_field in samples[0]:
        refs = [sample.get(ref_field, "") for sample in samples]
        metrics.update(compute_bleu_chrf(clean_preds, refs))
        term_acc = terminology_accuracy(clean_preds, samples, mode)
        term_cons = terminology_consistency(clean_preds, samples, mode)
        metrics["terminology_accuracy"] = term_acc
        metrics["terminology_consistency"] = term_cons

        print(
            f"[{lang}/{mode}] BLEU={fmt_metric(metrics['bleu'])} "
            f"chrF={fmt_metric(metrics['chrf'])} "
            f"term_acc={fmt_metric(term_acc['avg_ratio_pct'])}% "
            f"macro_cons={fmt_metric(term_cons['macro_avg_consistency'])} "
            f"weighted_cons={fmt_metric(term_cons['weighted_avg_consistency'])}"
        )
    else:
        print(f"[{lang}/{mode}] no reference field '{ref_field}' â€” metrics skipped")

    return {"predictions_file": str(pred_path), "metrics": metrics}


datasets = load_datasets(DATA_SPLIT, DATA_ROOT, MAX_SAMPLES)
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

summary = {
    "data_split": split_suffix(),
    "data_dir": str(DATA_ROOT),
    "prompt_examples_per_lang": {lang: len(SAMPLE_SENTENCES[lang]) for lang in LANG_GROUPS},
    "model": MODEL_NAME,
    "adapter_dir": str(ADAPTER_DIR),
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "learning_rate": LORA_LEARNING_RATE,
        "num_train_epochs": LORA_NUM_TRAIN_EPOCHS,
        "train_split": split_suffix(TRAIN_DATA_SPLIT),
        "max_train_samples": MAX_TRAIN_SAMPLES,
    },
    "max_samples": MAX_SAMPLES,
    "languages": {},
}

for lang in LANG_GROUPS:
    config = LANG_CONFIG[lang]
    samples = datasets[lang]
    lang_dir = OUTPUT_BASE / lang
    lang_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {lang}: {len(samples)} samples â†’ {config['target_lang']} ===")

    lang_results = {mode: run_mode(lang, mode, samples, lang_dir, config) for mode in MODES}
    summary["languages"][lang] = {
        "data_file": str(data_path(lang)),
        "sample_count": len(samples),
        **config,
        "modes": lang_results,
    }

metrics_path = OUTPUT_BASE / "metrics_summary.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nDone.", metrics_path)